# Universal Data Profiling Template
**UC20 — Enterprise Financial Data Platform | Sprint 2**

Use this notebook to profile any PostgreSQL staging table. Change `TABLE_NAME` for each assigned table. It reports data-quality patterns without assuming a business key, deleting duplicates, filling missing values, or modifying PostgreSQL.

**Safety:** this notebook runs read-only `SELECT` queries.

## 1. Imports and database connection

In [ ]:
import getpass
import re
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

DB_USER = "postgres"
DB_PASSWORD = getpass.getpass("PostgreSQL password: ")
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "efdp"

connection_url = URL.create(
    "postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)
engine = create_engine(connection_url)

with engine.connect() as conn:
    print("Connected to database:", conn.execute(text("SELECT current_database()")).scalar())

## 2. Choose the table

Change `TABLE_NAME` for each table. Examples might include `stg_ska1`, `stg_skat`, `stg_lfa1`, or `stg_t001`; use the actual names in your database.

In [ ]:
SCHEMA_NAME = "public"
TABLE_NAME = "stg_ska1"  # <-- CHANGE THIS for each assigned table

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", SCHEMA_NAME):
    raise ValueError("Invalid schema identifier")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", TABLE_NAME):
    raise ValueError("Invalid table identifier")

qualified_table = f'"{SCHEMA_NAME}"."{TABLE_NAME}"'

## 3. Confirm the table exists and inspect its schema

In [ ]:
schema_query = text("""
    SELECT column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_schema = :schema_name
      AND table_name = :table_name
    ORDER BY ordinal_position
""")

with engine.connect() as conn:
    schema_df = pd.read_sql(
        schema_query, conn,
        params={"schema_name": SCHEMA_NAME, "table_name": TABLE_NAME}
    )

if schema_df.empty:
    raise ValueError(f"Table {SCHEMA_NAME}.{TABLE_NAME} was not found or has no visible columns.")

display(schema_df)

## 4. Load table and inspect basic structure

In [ ]:
df = pd.read_sql(f"SELECT * FROM {qualified_table}", engine)

print(f"Table: {SCHEMA_NAME}.{TABLE_NAME}")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head())
display(df.dtypes.rename("pandas_dtype").to_frame())

## 5. Missing values and blank strings

In [ ]:
missing_profile = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
}).sort_values(["missing_percentage", "missing_count"], ascending=False)
display(missing_profile)

text_columns = df.select_dtypes(include=["object", "string"]).columns
blank_profile = (
    df[text_columns]
    .apply(lambda col: col.astype("string").str.strip().eq("").sum())
    .rename("blank_string_count")
    .to_frame()
)
display(blank_profile)

## 6. Exact duplicate analysis

In [ ]:
exact_duplicate_count = int(df.duplicated().sum())
duplicate_group_rows = int(df.duplicated(keep=False).sum())

print(f"Exact duplicate rows after first copies: {exact_duplicate_count:,}")
print(f"Rows belonging to exact-duplicate groups: {duplicate_group_rows:,}")
print(f"Exact duplicate percentage: {exact_duplicate_count / len(df) * 100:.2f}%" if len(df) else "Table is empty")

if exact_duplicate_count:
    display(df[df.duplicated(keep=False)].head(20))

## 7. Numeric-column profiling

In [ ]:
numeric_columns = df.select_dtypes(include="number").columns.tolist()
if numeric_columns:
    display(df[numeric_columns].describe().T)
else:
    print("No numeric columns detected by Pandas.")

## 8. Date-column profiling (optional)

Review the table schema and add actual date column names to `DATE_COLUMNS`.

In [ ]:
DATE_COLUMNS = []  # Example: ["document_date", "posting_date"]

for col in DATE_COLUMNS:
    if col not in df.columns:
        print(f"Skipping {col!r}: column not found.")
        continue

    parsed = pd.to_datetime(df[col], errors="coerce")
    invalid = df[col].notna() & parsed.isna()

    print(f"\n{col}")
    print("Missing values:", int(df[col].isna().sum()))
    print("Non-null values that failed parsing:", int(invalid.sum()))
    print("Earliest parsed date:", parsed.min())
    print("Latest parsed date:", parsed.max())

## 9. Optional candidate-key check

Do not assume every table's key. Fill `CANDIDATE_KEY` only after checking the data dictionary or SAP field definitions. An empty list skips the check.

In [ ]:
CANDIDATE_KEY = []  # Fill only after confirming the business key

if not CANDIDATE_KEY:
    print("Candidate-key check skipped: define CANDIDATE_KEY after confirming the table's business key.")
else:
    missing_key_columns = [col for col in CANDIDATE_KEY if col not in df.columns]
    if missing_key_columns:
        print("Key check skipped; columns not found:", missing_key_columns)
    else:
        null_key_rows = int(df[CANDIDATE_KEY].isna().any(axis=1).sum())
        duplicated_key_rows = int(df.duplicated(subset=CANDIDATE_KEY, keep=False).sum())
        key_group_sizes = df.groupby(CANDIDATE_KEY, dropna=False).size()

        print("Candidate key:", CANDIDATE_KEY)
        print("Rows with at least one null key component:", f"{null_key_rows:,}")
        print("Rows belonging to repeated-key groups:", f"{duplicated_key_rows:,}")
        print("Repeated key values:", f"{int((key_group_sizes > 1).sum()):,}")
        print("Maximum rows per key:", int(key_group_sizes.max()) if len(key_group_sizes) else 0)

        if duplicated_key_rows:
            display(
                df[df.duplicated(subset=CANDIDATE_KEY, keep=False)]
                .sort_values(CANDIDATE_KEY)
                .head(30)
            )

## 10. Export a compact profiling summary

In [ ]:
summary = pd.DataFrame([
    {"metric": "schema", "value": SCHEMA_NAME},
    {"metric": "table", "value": TABLE_NAME},
    {"metric": "row_count", "value": len(df)},
    {"metric": "column_count", "value": df.shape[1]},
    {"metric": "exact_duplicate_rows_excluding_first_copies", "value": int(df.duplicated().sum())},
    {"metric": "rows_belonging_to_exact_duplicate_groups", "value": int(df.duplicated(keep=False).sum())},
    {"metric": "columns_with_missing_values", "value": int(df.isna().any().sum())},
])

output_file = f"{TABLE_NAME}_profiling_summary.csv"
summary.to_csv(output_file, index=False)
print("Saved:", output_file)
display(summary)

## 11. Findings log

Complete after reviewing the outputs:

- **Table and purpose:**
- **Source row count / staging row count comparison:**
- **Missing-value findings:**
- **Blank-string findings:**
- **Exact duplicate findings:**
- **Candidate key and repeated-key findings:**
- **Date/numeric anomalies requiring business validation:**
- **Actions recommended:**
- **Fields that may be legitimately optional:**
- **Limitations / follow-up checks:**

**Rules:** do not automatically fill nulls with zero, delete duplicates, or modify staging tables. Verify source mappings and business semantics first. Document findings and create separate cleaned datasets only after the cleaning rule is justified.